In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import SimpleExpSmoothing
from statsmodels.tsa.api import ExponentialSmoothing
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import Holt
from sklearn.metrics import mean_squared_error
import warnings
from datetime import datetime
import calendar
warnings.filterwarnings("ignore")

def Monthly_days_forecasting(file_path, start_date_str=None, last_date_str=None):

  data=pd.read_csv(file_path)
  data["time_stamp"]=pd.to_datetime(data["time_stamp"],errors="coerce") #converting into datetime object
  data=data.dropna(subset=["time_stamp", "units"]) #dropping timestamp and units
  data=data.set_index("time_stamp") #timestamp is index now
  minute_level_data=data["units"].resample("min").sum() # converting to minute level
  day_level_data=data["units"].resample("D").sum() #using day level
  day_level_data.head()

  #fixing a empty day
  day_level_data=day_level_data[day_level_data>0]
  #removing a last day
  day_level_data=day_level_data[:-1]

  rolling_median = day_level_data.rolling(window=7, min_periods=1).median()
  sensor_failure_mask = day_level_data > rolling_median * 2.5
  day_level_data[sensor_failure_mask] = rolling_median[sensor_failure_mask]

  #Defining a Forecasting function name, check if day is from 0-07 apply SES, check if day is from 08,29 apply Holt, check if day is from 30-onward appply Holt winter.

  def forecasting_SES(day,steps):
    #Simple Exponential Smoothing model
    best_alpha=None
    best_mse=float("inf")
    for alpha in np.arange(0.1, 1.0, 0.1):
      model=SimpleExpSmoothing(day, initialization_method="estimated")
      model_fit=model.fit(smoothing_level=alpha,optimized=False)
      mse=mean_squared_error(day,model_fit.fittedvalues)
      if mse<best_mse:
        best_alpha=alpha
        best_mse=mse

    print(f"Best alpha: {best_alpha: .1f}")
    print(f"Best MSE: {best_mse: .4f}")
    print(f"Root mean squared error {np.sqrt(best_mse)}")
    # refit using best parameters found
    model=SimpleExpSmoothing(day, initialization_method="estimated")
    model_fit=model.fit(smoothing_level=best_alpha,optimized=False)

    # generate forecast with proper date index
    last_date = day.index[-1]
    forecast_index = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=steps,
    freq='D'
    )
    forecast = model_fit.forecast(steps=steps)
    forecast.index = forecast_index
    #forecasting printing
    print(f"Forecast using SES{forecast}")
    #plotting a graph
    # plt.figure(figsize=(15,10))
    plt.plot(day, color ="blue",label="Original values")
    plt.plot(model_fit.fittedvalues, color = "green", marker="o", label = "fitted values")
    plt.plot(forecast,color="red",marker="x",label="Forecasting values using SES")
    plt.title("Simple Exponential Smoothing")
    plt.xlabel("Time in days")
    plt.ylabel("Units")
    plt.legend()
    plt.grid(True)
    plt.show()
    print(day.head())
    # forecasting_SES(day) # Call the function here
    return forecast

  def forecasting_Holt(day,steps):
    best_alpha=None
    best_beta=None
    best_damping=None
    best_mse=float("inf")
    for alpha in np.arange(0.1, 1.0, 0.1):
      for beta in np.arange(0.01, 0.5, 0.01):
        for damping in [0.80,0.85,0.90,0.95,0.98]:
          model_double=Holt(day, damped_trend=True ,initialization_method="estimated")
          model_double_fit=model_double.fit(smoothing_level=alpha, smoothing_slope=beta, damping_slope=damping ,optimized=False)

          mse=mean_squared_error(day,model_double_fit.fittedvalues)
          if mse<best_mse:
            best_alpha=alpha
            best_beta=beta
            best_damping=damping
            best_mse=mse

    print(f"Best alpha: {best_alpha: .2f}")
    print(f"Best beta: {best_beta: .2f}")
    print(f"Best damping: {best_damping: .2f}")
    print(f"Best MSE: {best_mse: .4f}")
    print(f"Root mean squared error {np.sqrt(best_mse): .4f}")
    # refit using best parameters found
    best_model = Holt(day,
                      damped_trend=True,
                      exponential=False ,
                      initialization_method="estimated")
    best_fit = best_model.fit(
      smoothing_level=best_alpha,
      smoothing_slope=best_beta,
      damping_slope=best_damping,
      optimized=False
    )
    forecast = best_fit.forecast(steps=steps)  #generating forecast with proper date and time
    last_date = day.index[-1]
    forecast_index = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=steps,
    freq="D"
    )
    forecast.index = forecast_index
    print(f"Forecast Holt method values are: {forecast}")
    #graph
    plt.figure(figsize=(10,8))
    plt.plot(day, color ="blue",marker="o",label="Original values")
    plt.plot(best_fit.fittedvalues, color = "green", marker="o", label = "fitted values")
    plt.plot(forecast,color="red",marker="x",label="Forecasting values using Holt method")
    plt.title("Holt Method/Double Exponential Smoothing")
    plt.xlabel("Time in days")
    plt.ylabel("Units")
    plt.legend()
    plt.grid(True)
    plt.show()
    # forecasting_Holt(day)
    return forecast

    #Holt winter function
  def forecasting_Holt_Winter(day,steps):
    best_alpha=None
    best_beta=None
    best_gamma=None
    best_rmse=float("inf")
    for alpha in np.arange(0.1, 1.0, 0.1):
      for beta in np.arange(0.01, 0.5, 0.01):
        for gamma in np.arange(0.01, 0.5, 0.01):
          # In this Holt winter method i use seasonal_periods=5 depends on input data. Standard is seasonal_periods=7. Change it according to your data.
          model_triple=ExponentialSmoothing(day, trend="add", seasonal="add", damped_trend=True ,seasonal_periods=5 ,initialization_method="estimated")
          model_triple_fit=model_triple.fit(smoothing_level=alpha, smoothing_slope=beta, smoothing_seasonal=gamma, damping_slope=0.92 ,optimized=False)
          rmse=np.sqrt(mean_squared_error(day,model_triple_fit.fittedvalues))
          if rmse<best_rmse:
            best_alpha=alpha
            best_beta=beta
            best_gamma=gamma
            best_rmse=rmse

    print(f"Best alpha: {best_alpha: .1f}")
    print(f"Best beta: {best_beta: .1f}")
    print(f"Best gamma: {best_gamma: .1f}")
    print(f"Best RMSE: {best_rmse: .4f}")
    best_third_model=ExponentialSmoothing(day, trend="add", damped_trend=True , seasonal="add", seasonal_periods=5 ,initialization_method="estimated")
    best_third_fit=best_third_model.fit(smoothing_level=best_alpha, damping_slope=0.92,smoothing_slope=best_beta, smoothing_seasonal=best_gamma, optimized=False)

    forecast = best_third_fit.forecast(steps=steps)  #generating forecast with proper date and time
    last_date = day.index[-1]
    forecast_index = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=steps,
    freq="D"
    )
    forecast.index = forecast_index
    print(f"Forecast using Holt winter values are: {forecast}")

      #graph
    plt.figure(figsize=(10,8))
    plt.plot(day, color ="blue",marker="o",label="Original values")
    plt.plot(best_third_fit.fittedvalues, color = "green", marker="o", label = "fitted values")
    plt.plot(forecast,color="red",marker="x",label="Forecasting values using Holt winter")
    plt.title("Holt Winter Exponential Smoothing Method")
    plt.xlabel("Time in days")
    plt.ylabel("Units")
    plt.legend()
    plt.grid(True)
    plt.show()
    return forecast

  def forecasting_function(day,start_date_str,last_date_str):
    start_date=pd.to_datetime(start_date_str) #start_date
    last_date=pd.to_datetime(last_date_str)   #last_date
    month=start_date.month #month name
    year=start_date.year   #year name
    # days_in_month=calendar.monthrange(year,month)[1]  #month days
    month_name=calendar.month_name[month]   #month name
    day=day[(day.index>=start_date) & (day.index<=last_date)] #days from starting and ending date
    #remaining days
    last_data_date=day.index[-1]
    billing_end_date=pd.to_datetime(start_date_str) + pd.DateOffset(months=1)
    # month_end_date=pd.Timestamp(year=last_data_date.year, month=last_data_date.month, day=days_in_month)
    remaining_days=(billing_end_date - last_data_date).days
    # print(f"units consumed {units_consumed}")
    #units consumed so far
    units_consumed=day.sum()


    print(f"\n{"="*55}") #for just printing a line
    print(f"🗓️ Month: {month_name}")
    print(f"🗓️ Year: {year}")
    # print(f"🗓️ Days in month: {days_in_month}")
    print(f"🗓️ Starting date: {start_date_str}")
    print(f"🗓️ Last data date: {last_data_date}")
    print(f"🗓️ Billing end date: {billing_end_date}")
    print(f"🗓️ Remaining days: {remaining_days}")

    if remaining_days<=0:
      print(f"Month already completed, No need for forecasting")
      return

    if len(day) <=7:
      print("SES Method")
      forecast=forecasting_SES(day,steps=remaining_days)
    elif len(day) >=8 and len(day) <=20:
      print("Holts method")
      forecast=forecasting_Holt(day, steps=remaining_days)
    else:
      print("Holts winter method")
      forecast=forecasting_Holt_Winter(day,steps=remaining_days)
    forecasted_units=forecast.sum()
    total_monthly_units=units_consumed + forecasted_units
    print(f"🗓️ Units consumed so far: {units_consumed:.2f}")
    print(f"🗓️ Forecasted units: {forecasted_units:.2f}")
    print(f"🗓️ Total monthly units: {total_monthly_units:.2f}")

  if start_date_str is None:
    start_date_str=day_level_data.index.min().strftime("%y-%m-%d")
  if last_date_str is None:
    last_date_str=day_level_data.index.max().strftime("%y-%m-%d")
  forecasting_function(day_level_data,start_date_str,last_date_str)

import os
import glob

def watch_and_predict(folder_path, start_date_str=None,last_date_str=None):

    # Get all CSV files in folder
    all_files = glob.glob(folder_path + "/*.csv")

    if len(all_files) == 0:
        print("❌ No CSV files found in folder")
        return

    # Sort by newest file first
    all_files.sort(key=os.path.getmtime, reverse=True)
    newest_file = all_files[0]

    print(f"📂 Found {len(all_files)} files in folder")
    print(f"📄 Latest file: {newest_file}")
    print(f"🔄 Running prediction...")

    # Call your function
    result = Monthly_days_forecasting(newest_file, start_date_str, last_date_str)

    # return result

# Call it
watch_and_predict("/Data_Energy_Units",start_date_str="2026-04-08",last_date_str="2026-5-11")